# NumPy arrays: multidimensional computation

**Live notebook · approximately 45 minutes**

- Continue with the temperature grid from the first block
- Predict shapes before running code
- Use results and diagnostics to drive the spoken narrative

## Outcomes

After this block, you should be able to:

- apply element-wise operations without explicit Python loops;
- predict and diagnose broadcasting;
- reduce over intended axes;
- reshape, reorder, and combine arrays deliberately; and
- use a linear-algebra result and inspect its residuals.

## Setup

- Use the repository's default Pixi environment.
- The fixture axes remain `(time, y, x)`.

In [1]:
import numpy as np

temperature = np.load("temperature_grid.npy", allow_pickle=False)
times = ("06:00", "12:00")
x = np.linspace(0.0, 3.0, num=4)
y = np.arange(3, dtype=np.float64)

temperature.shape, times, x, y

((2, 3, 4), ('06:00', '12:00'), array([0., 1., 2., 3.]), array([0., 1., 2.]))

## Element-wise computation

- Arithmetic acts on every value.
- NumPy functions such as `abs`, `sqrt`, and `exp` also act element by element.
- Element-wise operations preserve shape.

In [2]:
temperature_kelvin = temperature + 273.15
distance_from_20 = np.abs(temperature - 20.0)

{
    "input shape": temperature.shape,
    "Kelvin shape": temperature_kelvin.shape,
    "first row in Kelvin": temperature_kelvin[0, 0],
    "largest distance from 20 °C": np.round(distance_from_20.max(), 2),
}

{'input shape': (2, 3, 4),
 'Kelvin shape': (2, 3, 4),
 'first row in Kelvin': array([291.35, 291.55, 291.85, 292.15]),
 'largest distance from 20 °C': np.float64(2.7)}

## Broadcasting

Compare shapes from the trailing axes:

- equal sizes are compatible;
- a size of `1` can expand;
- missing leading axes behave as size `1`.

In [3]:
x_bias = np.array([0.1, -0.1, 0.0, 0.2])
corrected = temperature - x_bias

temperature.shape, x_bias.shape, corrected.shape, corrected[0, 0]

((2, 3, 4), (4,), (2, 3, 4), array([18.1, 18.5, 18.7, 18.8]))

For `temperature - time_bias`, where `time_bias.shape == (2,)`:

- Does NumPy know that the two values describe time?
- Are the shapes compatible?

In [4]:
time_bias = np.array([0.0, 0.3])

try:
    temperature - time_bias
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

ValueError: operands could not be broadcast together with shapes (2,3,4) (2,) 


Insert length-one axes to express the intended alignment.

In [5]:
aligned_time_bias = time_bias[:, None, None]
corrected = temperature - aligned_time_bias

temperature.shape, aligned_time_bias.shape, corrected.shape

((2, 3, 4), (2, 1, 1), (2, 3, 4))

### Reconstruct the fixture

The prepared grid can now be read as a sequence of broadcasting decisions.

In [6]:
x_row = x[None, :]
y_column = y[:, None]

x_row.shape, y_column.shape, np.broadcast_shapes(x_row.shape, y_column.shape)

((1, 4), (3, 1), (3, 4))

In [7]:
morning = 18.2 + 0.25 * x_row - 0.45 * y_column
time_offset = np.array([0.0, 2.9])
generated = morning[None, :, :] + time_offset[:, None, None]
generated = np.round(generated, decimals=1)

generated.shape, np.array_equal(generated, temperature)

((2, 3, 4), True)

Broadcasting avoids manual repetition, but the computed result still occupies memory.

## Reductions and axes

- A reduction summarizes values.
- `axis` identifies the dimensions to remove.
- The remaining axes determine the result's meaning.

In [8]:
{
    "overall mean": temperature.mean(),
    "mean for each time": temperature.mean(axis=(1, 2)),
    "mean field shape": temperature.mean(axis=0).shape,
}

{'overall mean': np.float64(19.570833333333336),
 'mean for each time': array([18.125     , 21.01666667]),
 'mean field shape': (3, 4)}

For `temperature.mean(axis=(1, 2), keepdims=True)`:

- What is the result shape?
- Can it be subtracted from `temperature`?

In [9]:
spatial_mean = temperature.mean(axis=(1, 2), keepdims=True)
anomaly = temperature - spatial_mean

{
    "mean shape": spatial_mean.shape,
    "anomaly shape": anomaly.shape,
    "zero-mean check": np.allclose(anomaly.mean(axis=(1, 2)), 0.0),
}

{'mean shape': (2, 1, 1), 'anomaly shape': (2, 3, 4), 'zero-mean check': True}

## Reshape and reorder

- `reshape` groups the same values differently.
- `transpose` changes axis order.
- Neither operation implies independent storage.

In [10]:
spatial_vectors = temperature.reshape(temperature.shape[0], -1)

spatial_vectors.shape, np.shares_memory(temperature, spatial_vectors)

((2, 12), True)

In [11]:
time_last = temperature.transpose(1, 2, 0)

time_last.shape, np.shares_memory(temperature, time_last)

((3, 4, 2), True)

### Combine arrays

- `stack` creates a new axis.
- `concatenate` extends an existing axis.

Predict the shapes when a second run is added with `stack` or `concatenate`.

In [12]:
second_run = temperature + 0.5
parallel_runs = np.stack((temperature, second_run), axis=0)
longer_record = np.concatenate((temperature, second_run), axis=0)

temperature.shape, parallel_runs.shape, longer_record.shape

((2, 3, 4), (2, 2, 3, 4), (4, 3, 4))

## Linear algebra: separate coupled measurements

Two sensor channels influence one another:

- a matrix describes the coupling;
- a vector contains the measured temperatures;
- solve directly for the underlying temperatures.

The model is $R t = m$. Here $R$ is the sensor-response matrix, $t$ contains the two underlying temperatures, and $m$ contains the two measured temperatures.

In [13]:
response = np.array(
    [
        [0.90, 0.10],
        [0.05, 0.95],
    ]
)
measured = temperature[0, 0, :2]

response, measured

(array([[0.9 , 0.1 ],
        [0.05, 0.95]]),
 array([18.2, 18.4]))

Before running: what shape should the solution have, and is a matrix inverse needed?

In [14]:
corrected = np.linalg.solve(response, measured)

np.round(corrected, 3)

array([18.176, 18.412])

Substitute the result back into the model and inspect the mismatch.

In [15]:
reconstructed = response @ corrected
residual = reconstructed - measured

{
    "measured": measured,
    "reconstructed": np.round(reconstructed, 3),
    "maximum absolute residual": np.round(np.abs(residual).max(), 12),
}

{'measured': array([18.2, 18.4]),
 'reconstructed': array([18.2, 18.4]),
 'maximum absolute residual': np.float64(0.0)}

## Decision check

Before computing, ask:

1. What shape should the result have?
2. Which axes align, and which axes should disappear?
3. Am I creating a new axis or extending an existing one?
4. What relationship does the matrix encode?
5. Does substituting the result reproduce the measurements?

## Takeaways

- Element-wise operations preserve shape.
- Broadcasting follows shape, not scientific meaning.
- Reductions remove selected axes unless `keepdims=True`.
- `reshape`, `transpose`, `stack`, and `concatenate` answer different questions.
- Use `solve` for a square linear system and validate by substitution.

## Optional: fit a spatial trend step by step

**Optional extension · approximately 10–15 minutes**

Approximate the morning field by $T(x, y) \approx \beta_0 + \beta_x x + \beta_y y$. Here $T$ is temperature, $x$ and $y$ are spatial coordinates, and the three $\beta$ values are fitted coefficients.

### Give every observation coordinates

- `meshgrid` builds coordinate arrays from one-dimensional coordinates.
- With `indexing="xy"`, rows follow $y$ and columns follow $x$.

In [16]:
x_grid, y_grid = np.meshgrid(x, y, indexing="xy")

x_grid.shape, y_grid.shape, x_grid, y_grid

((3, 4),
 (3, 4),
 array([[0., 1., 2., 3.],
        [0., 1., 2., 3.],
        [0., 1., 2., 3.]]),
 array([[0., 0., 0., 0.],
        [1., 1., 1., 1.],
        [2., 2., 2., 2.]]))

### Make one row per observation

`ravel` turns each grid into a one-dimensional sequence with matching order.

In [17]:
x_values = x_grid.ravel()
y_values = y_grid.ravel()
observations = temperature[0].ravel()

x_values.shape, x_values[:4], y_values[:4], observations[:4]

((12,),
 array([0., 1., 2., 3.]),
 array([0., 0., 0., 0.]),
 array([18.2, 18.4, 18.7, 19. ]))

### Assemble the design matrix

- A column of ones represents $\beta_0$.
- The coordinate columns represent $\beta_x$ and $\beta_y$.
- `column_stack` places these columns side by side.

In [18]:
intercept = np.ones(observations.size)
design = np.column_stack((intercept, x_values, y_values))

design.shape, design[:5]

((12, 3),
 array([[1., 0., 0.],
        [1., 1., 0.],
        [1., 2., 0.],
        [1., 3., 0.],
        [1., 0., 1.]]))

### Fit more equations than unknowns

`lstsq` returns:

- the fitted coefficients;
- the residual sum of squares;
- the matrix rank; and
- singular values, which are not interpreted here.

In [19]:
coefficients, residual_sum_squares, rank, _ = np.linalg.lstsq(
    design, observations, rcond=None
)

{
    "coefficients [β₀, βₓ, βᵧ]": np.round(coefficients, 3),
    "residual sum of squares": np.round(residual_sum_squares[0], 4),
    "rank": rank,
}

{'coefficients [β₀, βₓ, βᵧ]': array([18.21 ,  0.243, -0.45 ]),
 'residual sum of squares': np.float64(0.0143),
 'rank': np.int32(3)}

### Reconstruct and validate

Use `@` to compute fitted values, then restore the grid shape and inspect residuals.

In [20]:
fitted_values = design @ coefficients
fitted_field = fitted_values.reshape(temperature[0].shape)
residual = temperature[0] - fitted_field

{
    "fitted shape": fitted_field.shape,
    "maximum absolute residual": np.round(np.abs(residual).max(), 3),
    "generator coefficients": np.array([18.2, 0.25, -0.45]),
    "fitted coefficients": np.round(coefficients, 3),
}

{'fitted shape': (3, 4),
 'maximum absolute residual': np.float64(0.06),
 'generator coefficients': array([18.2 ,  0.25, -0.45]),
 'fitted coefficients': array([18.21 ,  0.243, -0.45 ])}

The small residual reflects rounding in this constructed dataset. A good fit does not by itself establish that a spatial plane is an adequate model for real measurements.